# Polarized Trees — Benchmark, Demo, and Inference

This notebook has **two roles**:

- it is a simple end-to-end example of how to use `PolarizedTreesBenchmark`;
- it is also the reproducibility code used for the synthetic benchmark reported in the paper.

The workflow is deliberately kept in one place:

**fixed synthetic data → hyperparameter search → best configuration → recovery evaluation → unseen inference → F/C/P**

The benchmark module performs the actual model selection. The notebook does not manually rank configurations or reconstruct the selected model.

> **Important:** the synthetic corpora must be generated first with `datasetdemo.ipynb`. This notebook reuses those fixed datasets so every configuration is evaluated on exactly the same data.


## What is fixed and what can be changed?

This notebook uses the **default benchmark settings provided by `polartox.benchmark`**. These defaults give a ready-to-run starting point, but they are not required.

The main benchmark settings can be changed:

- **search space** — pass a dictionary defining the values to consider for each hyperparameter;
- **strategy** — `"full"` evaluates every configuration, while `"random"` samples configurations;
- **number of runs** — controls how many configurations are sampled with random search;
- **seed** — makes random search reproducible;
- **metrics** — choose which recovery metrics to calculate;
- **selection metric** — choose which metric determines the best configuration;
- **selection direction** — choose `"max"` or `"min"`;
- **pipeline settings** — dimensions, scale, and any fixed pipeline parameters.

For this notebook, the default search space, metrics, and selection metric are imported directly from the package. You can replace `DEFAULT_SEARCH_SPACE` with your own dictionary without changing the benchmark code.

Ground truth is required for recovery-based model selection. After the best configuration is selected, the resulting pipeline can be run **without ground truth**, which is the normal inference setting for real annotation data.


## 1. Imports and experiment settings

Only a small amount of configuration is needed here. The actual search, evaluation, ranking, and selection are delegated to `PolarizedTreesBenchmark`.


In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

from polartox.benchmark import (
    PolarizedTreesBenchmark,
    DEFAULT_SEARCH_SPACE,
    DEFAULT_METRICS,
    DEFAULT_SELECTION_METRIC,
)
from polartox.pipeline import PolarizedTreesPipeline


# Locate the repository root.
CWD = Path.cwd().resolve()

if (CWD / "benchmark_config.py").exists():
    PROJECT_ROOT = CWD
elif (CWD.parent / "benchmark_config.py").exists():
    PROJECT_ROOT = CWD.parent
else:
    raise FileNotFoundError(
        "Could not locate project root. "
        "Run this notebook from the notebooks directory "
        "or the project root."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


from benchmark_config import (
    DIMS,
    SCALE,
    BENCHMARK_CORPORA,
    INFERENCE_CORPUS,
)


DATA_DIR = PROJECT_ROOT / "benchmark_data"
RESULTS_DIR = PROJECT_ROOT / "benchmark_results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Paper settings.
N_RUNS = 800
SEED = 0

print("Project root:", PROJECT_ROOT)
print("Benchmark corpora:", BENCHMARK_CORPORA)
print("Inference corpus:", INFERENCE_CORPUS)
print("Random configurations:", N_RUNS)
print("Seed:", SEED)


## 2. Load the fixed synthetic datasets

The first notebook generates these datasets once and stores both the annotations and the per-text ground truth.

Ground truth tells us which SCD dimensions actually generated the polarization. That makes recovery metrics possible and allows the benchmark to perform model selection.

The inference corpus is kept separate and is not used for selecting the configuration.


In [ ]:
def load_corpus(name):
    dataset = pd.read_csv(
        DATA_DIR / f"{name}_dataset.csv"
    )

    with open(
        DATA_DIR / f"{name}_ground_truth.json",
        encoding="utf-8",
    ) as f:
        ground_truth = {
            int(text_id): value
            for text_id, value in json.load(f).items()
        }

    return dataset, ground_truth


corpora = {
    name: load_corpus(name)
    for name in BENCHMARK_CORPORA
}

inference_dataset, inference_ground_truth = load_corpus(
    INFERENCE_CORPUS
)


for name, (dataset, ground_truth) in corpora.items():
    print(
        f"{name}: "
        f"{dataset.shape[0]:,} annotations | "
        f"{dataset['text_id'].nunique()} texts | "
        f"{len(ground_truth)} ground-truth entries"
    )

print(
    f"{INFERENCE_CORPUS}: "
    f"{inference_dataset.shape[0]:,} annotations | "
    f"{inference_dataset['text_id'].nunique()} texts"
)


## 3. Prepare the model-selection data

`PolarizedTreesBenchmark` works on one annotation dataset. We therefore combine A/B/C into one development corpus.

The text IDs are shifted so that the three corpora remain completely distinct. The three corpora contain the same number of texts, so their combined mean gives each corpus equal weight.

This is only a data-preparation step; it does not change the annotations or their ground truth.


In [ ]:
text_counts = {
    name: dataset["text_id"].nunique()
    for name, (dataset, _) in corpora.items()
}

if len(set(text_counts.values())) != 1:
    raise ValueError(
        "A/B/C must contain the same number of texts "
        "for equal-weight mean Jaccard."
    )


benchmark_parts = []
benchmark_ground_truth = {}

offset = 0

for corpus_name in BENCHMARK_CORPORA:
    dataset, ground_truth = corpora[corpus_name]

    part = dataset.copy()

    text_ids = sorted(
        part["text_id"].unique()
    )

    id_map = {
        old_id: offset + i
        for i, old_id in enumerate(text_ids)
    }

    part["text_id"] = part["text_id"].map(id_map)

    benchmark_parts.append(part)

    for old_id, config in ground_truth.items():
        benchmark_ground_truth[
            id_map[old_id]
        ] = config

    offset += len(text_ids)


benchmark_dataset = pd.concat(
    benchmark_parts,
    ignore_index=True,
)

print("Combined annotations:", benchmark_dataset.shape)
print("Combined texts:", benchmark_dataset["text_id"].nunique())
print("Ground-truth entries:", len(benchmark_ground_truth))


## 4. Define the paper's search space

For the paper, we consider the full valid search space of 3,240 configurations.

The space is written as explicit configurations rather than a simple Cartesian product because `beta` is meaningful only when `variant="beta"`.

For a smaller demonstration, this cell can be replaced by a small custom search space. The benchmark API supports both forms.


In [ ]:
SEARCH_SPACE = DEFAULT_SEARCH_SPACE.copy()

print("Search space:")
for parameter, values in SEARCH_SPACE.items():
    print(f"  {parameter}: {values}")

n_configurations = 1
for values in SEARCH_SPACE.values():
    n_configurations *= len(values)

print("\nTotal possible configurations:", n_configurations)


## 5. Create the base Polarized Trees pipeline

The user provides **one pipeline object** to the benchmark.

The benchmark uses its settings as the base and creates candidate pipelines internally while testing configurations. After model selection, the selected pipeline is available directly through `benchmark.get_best_pipeline()`.

The user therefore does not need to manually reconstruct the winning pipeline.


In [ ]:
pipeline = PolarizedTreesPipeline(
    dims=DIMS,
    scale=SCALE,
)

print(pipeline)


## 6. Run model selection

This is the main benchmark step.

The benchmark:

1. generates the configurations from the supplied search-space dictionary;
2. evaluates the requested configurations on the same annotations and ground truth;
3. computes the requested recovery metrics;
4. ranks the configurations using the selected metric;
5. stores the best configuration and the corresponding pipeline.

Here we use random search so that the same notebook can also demonstrate the `random` strategy. For a small search space, `strategy="full"` is useful because it evaluates every possible configuration.

The defaults are imported from the package, but all of these choices can be changed.


In [ ]:
benchmark = PolarizedTreesBenchmark(
    pipeline=pipeline,
    annotations=benchmark_dataset,
    ground_truth=benchmark_ground_truth,

    # Paper search.
    search_space=SEARCH_SPACE,
    strategy="random",
    n_runs=N_RUNS,
    seed=SEED,

    # Recovery metrics available in the synthetic setting.
    metrics=DEFAULT_METRICS,

    # Paper selection criterion.
    selection_metric=DEFAULT_SELECTION_METRIC,

    # Keep the highest score.
    selection_direction="max",

    top_k=20,
    verbose=True,
)

benchmark.run()


## 7. Inspect the selected configuration

The benchmark now contains the complete model-selection result.

Nothing is selected manually here. These values are returned by the benchmark after ranking all evaluated configurations.


In [ ]:
print("Best configuration:")
print(benchmark.get_best_config())

print("\nBest mean Jaccard:")
print(benchmark.get_best_score())

print("\nTop configurations:")
display(benchmark.get_top_configs())

## 8. Inspect all benchmark results

The complete table contains one row per evaluated configuration.

This is useful for checking how sensitive performance is to the different hyperparameters and for reproducing the ranking reported in the paper.


In [ ]:
results_df = benchmark.results

print("Evaluated configurations:", len(results_df))

display(
    results_df.head(20)
)


## 9. Get the selected pipeline

The benchmark stores the winning pipeline itself.

This is the pipeline that should be used for the subsequent evaluation and inference steps. There is no need to instantiate another `PolarizedTreesPipeline` manually.


In [ ]:
best_pipeline = benchmark.get_best_pipeline()

print(best_pipeline)


## 10. Evaluate the selected configuration on A/B/C

Model selection was performed on the combined development data. We now report the selected configuration separately on the three benchmark distributions.

Because ground truth is available, we can report Jaccard, precision, recall, and exact match.


In [ ]:
selected_rows = []

for corpus_name in BENCHMARK_CORPORA:
    dataset, ground_truth = corpora[corpus_name]

    output = best_pipeline.run_full_evaluation(
        dataset,
        ground_truth=ground_truth,
        verbose=False,
    )

    recovery = output["recovery"]

    selected_rows.append({
        "corpus": corpus_name,
        "jaccard": recovery["jaccard"].mean(),
        "precision": recovery["precision"].mean(),
        "recall": recovery["recall"].mean(),
        "exact_match": recovery["exact_match"].mean(),
    })


selected_recovery = pd.DataFrame(
    selected_rows
)

mean_row = pd.DataFrame([{
    "corpus": "Mean",
    "jaccard": selected_recovery["jaccard"].mean(),
    "precision": selected_recovery["precision"].mean(),
    "recall": selected_recovery["recall"].mean(),
    "exact_match": selected_recovery["exact_match"].mean(),
}])

selected_recovery = pd.concat(
    [selected_recovery, mean_row],
    ignore_index=True,
)

display(selected_recovery)


## 11. Run inference without ground truth

This is the important transition from **validation** to **inference**.

The selected configuration is now fixed. We deliberately do not pass ground truth.

In this setting Polarized Trees reports the outputs that are available on real annotation data:

- **F** — dimension frequency across tree depths;
- **C** — subgroup pole consistency;
- **P** — subgroup polarization reduction;
- diagnostics describing the resulting trees and residual polarization.

The ground-truth recovery metrics are not part of this inference result.


In [ ]:
inference_results = best_pipeline.run_full_evaluation(
    inference_dataset,
    ground_truth=None,
    verbose=True,
)

F = inference_results["F"]
C = inference_results["C"]
P = inference_results["P"]
diagnostics = inference_results["diagnostics"]


print("=== F: Dimension Frequency ===")
display(F)

print("=== C: Subgroup Pole Consistency ===")
display(C.head(10))

print("=== P: Subgroup PRG ===")
display(P.head(10))

print("=== Diagnostics ===")
display(
    pd.Series(
        diagnostics,
        name="value",
    )
)


## 12. Save the benchmark and inference results

The benchmark report records the search settings, selected configuration, score, and top configurations.

The additional files provide the paper-facing recovery and inference outputs.


In [ ]:
# Complete benchmark outputs.
benchmark.save_results(
    RESULTS_DIR / "benchmark_results.csv"
)

benchmark.save_report(
    RESULTS_DIR / "benchmark_report.json"
)


# Selected configuration and recovery on A/B/C.
with open(
    RESULTS_DIR / "selected_configuration.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        benchmark.best_config,
        f,
        indent=2,
    )

selected_recovery.to_csv(
    RESULTS_DIR / "selected_configuration_recovery.csv",
    index=False,
)


# Ground-truth-free inference outputs.
F.to_csv(
    RESULTS_DIR / "fcp_F_dimension_frequency.csv"
)

C.to_csv(
    RESULTS_DIR / "fcp_C_pole_consistency.csv"
)

P.to_csv(
    RESULTS_DIR / "fcp_P_subgroup_prg.csv"
)


print(
    "Saved results to:",
    RESULTS_DIR.resolve()
)


## 13. Adapting this notebook to another experiment

The notebook uses the package defaults as a convenient starting point. The benchmark itself is deliberately configurable.

The main things to change are:

```text
search_space        → a dictionary of hyperparameter values to search
strategy            → "full" or "random"
n_runs              → number sampled for random search
seed                → reproducible random sampling
metrics             → recovery metrics to compute
selection_metric    → metric used to choose the winner
selection_direction → "max" or "min"
pipeline            → fixed dimensions, scale, and other settings
annotations         → your annotation dataset
ground_truth        → required for recovery-based model selection
```

For example, a custom search can be as simple as:

```python
SEARCH_SPACE = {
    "relative_h": [True],
    "h": [0.05, 0.10, 0.15],
    "max_depth": [4, 6],
}
```

You then pass that dictionary directly to `PolarizedTreesBenchmark`.

If you already have an annotation dataset, you can use it directly as long as it follows the expected annotation format. Synthetic data is only needed when you want known ground truth for objective recovery evaluation.

If you do not have ground truth, the benchmark cannot use recovery metrics for model selection. In that case, use `PolarizedTreesPipeline` directly in inference mode and interpret F/C/P and diagnostics.

## Next steps

1. Run `datasetdemo.ipynb` to generate the fixed synthetic datasets.
2. Run this notebook to benchmark configurations and select the best pipeline.
3. Inspect the saved benchmark and inference outputs.
4. For another experiment, change the search-space dictionary and/or benchmark settings and rerun.
